In [54]:
import numpy as np
import scipy
import json
import matplotlib.pyplot as plt
from datetime import datetime
from qiskit.quantum_info import Pauli

np.set_printoptions(linewidth=200, precision=4, suppress=True)

In [20]:
# Initialize Pauli matrices

pauli_x = Pauli('X')
pauli_y = Pauli('Y')
pauli_z = Pauli('Z')
identity = Pauli('I')

X = pauli_x.to_matrix()
Y = pauli_y.to_matrix()
Z = pauli_z.to_matrix()
I = identity.to_matrix()

Ix1 = np.kron(X/2,I)
Iy1 = np.kron(Y/2,I)
Iz1 = np.kron(Z/2,I)
Ix2 = np.kron(I,X/2)
Iy2 = np.kron(I,Y/2)
Iz2 = np.kron(I,Z/2)

J = 696

In [33]:
U_cnot = np.array([[1,0,0,0],
                   [0,1,0,0],
                   [0,0,0,1],
                   [0,0,1,0]])

U_perm = np.array([[1,0,0,0],
                   [0,0,0,1],
                   [0,1,0,0],
                   [0,0,1,0]])

Unitary matrix --> Hamiltonian and pulse parameters = difficult

Pulse parameters --> Unitary matrix = easy

Method: change pulse parameters, calculate how similar the resulting Unitary matrix is to Target unitary matrix, compare and optimize

In [21]:
# Parameters for a 2 qubit Rx(pi/2) pulse, Hydrogen and Phosphorus

R1=R2=1
w_rf=6250
phi_rf=np.pi/2
t_rf=40e-6

w0=w_rf
w1_H=w1_P=6250


In [22]:
# 2 qubit Unitary matrix calculation from system Hamiltonian

def twoqubit_unitary(R1,R2,w_rf,phi_rf,t_rf,w0,w1_H,w1_P):
    H0 = 2*np.pi*(w0-w_rf)*Iz1+2*np.pi*(w0-w_rf)*Iz2

    HJ = 2*np.pi*J*(np.kron(Z,Z)/4)

    H1 = 2*np.pi*R1*w1_H*(np.cos(phi_rf)*Ix1+np.sin(phi_rf)*Iy1)+2*np.pi*R2*w1_P*(np.cos(phi_rf)*Ix2+np.sin(phi_rf)*Iy2)

    return scipy.linalg.expm(-1j*(H0+HJ+H1)*t_rf)



In [23]:
Rypi2 = twoqubit_unitary(R1,R2,w_rf,phi_rf,t_rf,w0,w1_H,w1_P)

In [24]:
rho = Iz1

rho_f = Rypi2 @ rho @ Rypi2.conj().T

rho_f

array([[ 0.0003+0.j    ,  0.    -0.j    ,  0.4993-0.0219j, -0.    -0.0139j],
       [ 0.    +0.j    ,  0.0003+0.j    , -0.    -0.0139j,  0.4993+0.0219j],
       [ 0.4993+0.0219j, -0.    +0.0139j, -0.0003+0.j    ,  0.    +0.j    ],
       [-0.    +0.0139j,  0.4993-0.0219j,  0.    -0.j    , -0.0003+0.j    ]])

In [25]:
print(np.trace(rho_f @ Ix1))
print(np.trace(rho_f @ Iy1))
print(np.trace(rho_f @ Iz1))


(0.9986568311717521+0j)
(-6.591949208711867e-17+0j)
(0.000608575287973024+0j)


1. Create unitary operator

$$
U = R_{-z}^2\left(\frac{\pi}{4}\right)
    R_{-y}^2\left(\frac{\pi}{4}\right)
    R_{z}^1\left(\frac{\pi}{4}\right)
    R_{y}^1\left(\frac{\pi}{4}\right)
$$

In [26]:
# Rotation Rz through euler angles Rz(theta)=Rx(pi/2)Ry(theta)Rx(-pi/2)

R1 = twoqubit_unitary(0,1,6250,0,40e-6,6250,6250,6250)
R2 = twoqubit_unitary(0,1,6250,-np.pi/2,20e-6,6250,6250,6250)
R3 = twoqubit_unitary(0,1,6250,np.pi,40e-6,6250,6250,6250)

R2_zpi4 = R1 @ R2 @ R3

print(R2_zpi4)



[[ 0.9409+0.3308j  0.0729-0.j      0.    +0.j      0.    +0.j    ]
 [-0.0729-0.j      0.9409-0.3308j  0.    +0.j      0.    +0.j    ]
 [ 0.    +0.j      0.    +0.j      0.8983+0.4334j -0.0724-0.j    ]
 [ 0.    +0.j      0.    +0.j      0.0724+0.j      0.8983-0.4334j]]


In [27]:
# Rotation Ry

R2_ypi4 = twoqubit_unitary(0,1,6250,-np.pi/2,20e-6,6250,6250,6250)

In [28]:
# Rotation Rz through euler angles Rz(theta)=Rx(pi/2)Ry(theta)Rx(-pi/2)

R1 = twoqubit_unitary(1,0,6250,0,40e-6,6250,6250,6250)
R2 = twoqubit_unitary(1,0,6250,np.pi/2,20e-6,6250,6250,6250)
R3 = twoqubit_unitary(1,0,6250,np.pi,40e-6,6250,6250,6250)

R1zpi4 = R1 @ R2 @ R3

print(R1zpi4)


[[ 0.8983-0.4334j  0.    +0.j      0.0724-0.j      0.    +0.j    ]
 [ 0.    +0.j      0.9409-0.3308j  0.    +0.j     -0.0729-0.j    ]
 [-0.0724+0.j      0.    +0.j      0.8983+0.4334j  0.    +0.j    ]
 [ 0.    +0.j      0.0729-0.j      0.    +0.j      0.9409+0.3308j]]


In [29]:
# Rotation Ry

R1ypi4 = twoqubit_unitary(1,0,6250,np.pi/2,20e-6,6250,6250,6250)

U achieved using hard pulses

In [30]:
U = R2_zpi4 @ R2_ypi4 @ R1zpi4 @ R1ypi4

print(U)

[[ 0.8392-0.1136j  0.3991-0.0133j -0.2826+0.0629j -0.1991-0.j    ]
 [-0.3088+0.2879j  0.6379-0.4843j  0.0969-0.107j  -0.3259+0.2307j]
 [ 0.1646+0.2581j  0.0969+0.107j   0.5435+0.7115j  0.1871+0.221j ]
 [-0.1047-0.j      0.4162-0.0706j -0.305 +0.0258j  0.8392-0.1136j]]


Strongly Modulated Pulse optimization

In [ ]:
# SMP following SpinQ manual 

# constants
n = 2
w0 = 6250
w1_max = 6250

# Compute fidelity aka loss function

def fidelity(R, phi, omega_rf, dts, U_target):
    U = np.eye(2**n)
    for k in range(N):
        
        H_0 = 2*np.pi*(w0-omega_rf[k])*Iz1+2*np.pi*(w0-omega_rf[k])*Iz2
        H_j = 2*np.pi*J*(np.kron(Z,Z)/4)
        H_rf = 2*np.pi*R[k]*w1_max*(np.cos(phi[k])*Ix1+np.sin(phi[k])*Iy1)+2*np.pi*R[k]*w1_max*(np.cos(phi[k])*Ix2+np.sin(phi[k])*Iy2)
        
        H_total = H_0 + H_j + H_rf
        
        U = scipy.linalg.expm(-1j*H_total*dts[k]) @ U
        
    return np.abs(np.trace(U_target.conj().T @ U)) / (2**n)

# Gradient descent algorithm

# Calculate numerical gradient

def num_grad(R, phi, omega_rf, dts, U_target, d=1e-2 ):
    grad_R = np.zeros_like(R)
    grad_phi = np.zeros_like(phi)
    grad_omega = np.zeros_like(omega_rf)

    eps = 1e-6
    # per-component increments
    dR = np.maximum(d*R, eps)
    dphi = np.maximum(d*np.abs(phi), eps)
    dw = np.maximum(d*omega_rf, eps)

    Q0 = 1 - fidelity(R, phi, omega_rf, dts, U_target)

    for k in range(N):
        R_d = R.copy()
        R_d[k] += dR[k]
        grad_R[k] = (1-fidelity(R_d, phi, omega_rf, dts, U_target) - Q0)/dR[k]

        phi_d = phi.copy()
        phi_d[k] += dphi[k]
        grad_phi[k] = (1-fidelity(R, phi_d, omega_rf, dts, U_target) - Q0)/dphi[k]

        omega_d = omega_rf.copy()
        omega_d[k] += dw[k]
        grad_omega[k] = (1-fidelity(R, phi, omega_d, dts, U_target) - Q0)/dw[k]

    return grad_R, grad_phi, grad_omega

# Save to json file

def export_to_json_SMP(filename, title, date, fidelity, totalpulsewidth, slices, , owner="Unkown")


In [32]:
# Gradient descent loop

U_target = U

J = 696
T = 200e-6
N = 50  # maybe add more divisions?
dt = T/N

# initialize
R = np.full(N, 1.0, dtype=float)
phi = np.zeros(N, dtype=float)
omega_rf = np.full(N, w0, dtype=float)
dts = np.full(N, dt, dtype=float)

learn_rate = 0.05 # 500 iterations 0.01 is small - 38 minutes - fidelity = 0.7244; 500 iterations 0.05 - plateau at fidelity = 0.74
iter = 200

for i in range(iter):
    grad_R, grad_phi, grad_omega = num_grad(R, phi, omega_rf, dts, U_target)
    
    R -= learn_rate * grad_R
    phi -= learn_rate * grad_phi
    omega_rf -= learn_rate * grad_omega
    
    R = np.clip(R,0,1)
    phi = phi % (2*np.pi)
    
    if i % 10 == 0 or i == iter-1:
        print(f"Iteration {i}, Fidelity={fidelity(R, phi, omega_rf, dts, U_target):.6f}")

Iteration 0, Fidelity=0.258031
Iteration 10, Fidelity=0.335646
Iteration 20, Fidelity=0.417444
Iteration 30, Fidelity=0.495428
Iteration 40, Fidelity=0.562964
Iteration 50, Fidelity=0.616736
Iteration 60, Fidelity=0.656741
Iteration 70, Fidelity=0.685020
Iteration 80, Fidelity=0.704293
Iteration 90, Fidelity=0.717103
Iteration 100, Fidelity=0.725476
Iteration 110, Fidelity=0.730889
Iteration 120, Fidelity=0.734364
Iteration 130, Fidelity=0.736584
Iteration 140, Fidelity=0.738000
Iteration 150, Fidelity=0.738900
Iteration 160, Fidelity=0.739473
Iteration 170, Fidelity=0.739837
Iteration 180, Fidelity=0.740070
Iteration 190, Fidelity=0.740218
Iteration 199, Fidelity=0.740306


In [ ]:
# to json

In [36]:
U_target = U_cnot

J = 696
T = 200e-6
N = 50  # maybe add more divisions?
dt = T/N

# initialize
R = np.full(N, 1.0, dtype=float)
phi = np.zeros(N, dtype=float)
omega_rf = np.full(N, w0, dtype=float)
dts = np.full(N, dt, dtype=float)

learn_rate = 0.05 # 500 iterations 0.01 is small - 38 minutes - fidelity = 0.7244; 500 iterations 0.05 - plateau at fidelity = 0.74
iter = 200

for i in range(iter):
    grad_R,grad_phi,grad_omega = num_grad(R, phi, omega_rf, dts, U_target) #,grad_dt
    
    R -= learn_rate * grad_R
    phi -= learn_rate * grad_phi
    omega_rf -= learn_rate * grad_omega
    #dts -= learn_rate * grad_dt
    
    R = np.clip(R,0,1)
    phi = phi % (2*np.pi)
    
    if i % 10 == 0 or i == iter-1:
        print(f"Iteration {i}, Fidelity={fidelity(R, phi, omega_rf, dts, U_target):.6f}")

Iteration 0, Fidelity=0.352359
Iteration 10, Fidelity=0.385224
Iteration 20, Fidelity=0.412809
Iteration 30, Fidelity=0.435352
Iteration 40, Fidelity=0.453416
Iteration 50, Fidelity=0.467692
Iteration 60, Fidelity=0.478872
Iteration 70, Fidelity=0.487583
Iteration 80, Fidelity=0.494357
Iteration 90, Fidelity=0.499630
Iteration 100, Fidelity=0.503748
Iteration 110, Fidelity=0.506982
Iteration 120, Fidelity=0.509543
Iteration 130, Fidelity=0.511589
Iteration 140, Fidelity=0.513244
Iteration 150, Fidelity=0.514599
Iteration 160, Fidelity=0.515724
Iteration 170, Fidelity=0.516671
Iteration 180, Fidelity=0.517480
Iteration 190, Fidelity=0.518180
Iteration 199, Fidelity=0.518737


GRAPE Algorithm

In [59]:
# Gradient Ascent - GRAPE

# a bit different from the basic SMP, the parameter phi is omitted, collapsed into the amplitude which now differs for each orthogonal direction x, y

# constants
n = 2
d = 2**n
w0 = 6250
w1_max = 6250

def p_calc(R_x, R_y, U_target):
    P_list = []
    P = U_target.copy()
    H_x = 2*np.pi*w1_max*(Ix1+Ix2)
    H_y = 2*np.pi*w1_max*(Iy1+Iy2)
    H_J = 2*np.pi*J*(np.kron(Z,Z)/4)
    for k in reversed(range(N)):
        H = H_J + R_x[k]*H_x + R_y[k]*H_y
        Uk = scipy.linalg.expm(-1j * H * dt)
        P = Uk.conj().T @ P
        P_list.insert(0, P)

    return P_list

def x_calc(R_x,R_y):
    X_list = []
    U = np.eye(d, dtype=complex)
    H_x = 2*np.pi*w1_max*(Ix1+Ix2)
    H_y = 2*np.pi*w1_max*(Iy1+Iy2)
    H_J = 2*np.pi*J*(np.kron(Z,Z)/4)
    for k in range(N):
        H = H_J + R_x[k]*H_x + R_y[k]*H_y
        Uk = scipy.linalg.expm(-1j * H * dt)
        U = Uk @ U
        X_list.append(U)

    return X_list, U

def fidelity_grape(U):
    return np.abs(np.trace(U_target.conj().T @ U))/d

def grape_grad(R_x, R_y, U_target):
    
    X_list, U_final = x_calc(R_x, R_y)
    P_list = p_calc(R_x, R_y, U_target)
    
    H_x = 2*np.pi*w1_max*(Ix1+Ix2)
    H_y = 2*np.pi*w1_max*(Iy1+Iy2)
    H_J = 2*np.pi*J*(np.kron(Z,Z)/4)
    
    phi = np.trace(U_target.conj().T @ U_final)
    F0 = fidelity_grape(U_final)
    
    grad_x = np.zeros((N,1))
    grad_y = np.zeros((N,1))
    
    for k in range(N):
        Xk = X_list[k]
        Pk = P_list[k]
        
        dX = -1j * dt * H_x @ Xk
        dY = -1j * dt * H_y @ Xk
        
        grad_x[k] = (2/d) * np.real(
            np.trace(Pk.conj().T @ dX) * np.conj(phi)
        )
        
        grad_y[k] = (2/d) * np.real(
            np.trace(Pk.conj().T @ dY) * np.conj(phi)
        )
        
    return F0, grad_x, grad_y, U_final

def export_to_json_GRAPE(filename, title, fidelity, totalpulsewidth, slices, R_x, R_y, owner="Unkown"):
    
    R_x = np.asarray(R_x).flatten()
    R_y = np.asarray(R_y).flatten()
    
    dt = totalpulsewidth / slices

    channel1_pulses = []
    channel2_pulses = []

    for k in range(N):

        rx = float(R_x[k])
        ry = float(R_y[k])

        amplitude = np.sqrt(rx**2 + ry**2)
        phase = np.degrees(np.arctan2(ry, rx))  # convert to degrees

        pulse_entry = {
            "detuning": 0,
            "phase": float(phase),
            "amplitude": float(amplitude),
            "width": float(dt)
        }

        # Same pulse on both channels (modify if needed)
        channel1_pulses.append(pulse_entry)
        channel2_pulses.append(pulse_entry.copy())
        
        data = {
        "description": {
            "TITLE": title,
            "OWNER": owner,
            "DATE": datetime.now().strftime("%d-%m-%Y"),
            "FIDELITY": float(fidelity),
            "TOTALPULSEWIDTH": float(totalpulsewidth),
            "SLICES": int(N)
        },
        "parameters": {
            "offset": [
                {
                    "channel1_pulsefre_offset": 0,
                    "channel1_framefre_offset": 0,
                    "channel2_pulsefre_offset": 0,
                    "channel2_framefre_offset": 0
                }
            ]
        },
        "pulse": {
            "channel1_pulse": channel1_pulses,
            "channel2_pulse": channel2_pulses
        }
    }

    with open(filename, "w") as f:
        json.dump(data, f, indent=4)

    print(f"Pulse file saved to {filename}")
    

    # --- Optional: monotonicity check (important!) ---
    F_new, _, _, _ = grape_grad(R_x_new, R_y_new, U_target)

    if F_new >= F:
        R_x = R_x_new
        R_y = R_y_new
        F = F_new
    else:
        # reduce step size if fidelity decreases
        learn_rate *= 0.5

In [58]:
# learning loop

np.random.seed(1234)
U_target = U_cnot
print("Target matrix: \n", U_target)

J = 696
T = 200e-6
N = 100  # maybe add more divisions?
dt = T/N

# initialize
R_x = 0.2 * np.random.rand(N,1)
R_y = 0.2 * np.random.rand(N,1)

learn_rate = 0.01
max_iter = 5000
fid_history = []

for i in range(max_iter):

    F, grad_x, grad_y, Ufinal = grape_grad(R_x, R_y, U_target)

    # update
    R_x_new = R_x + learn_rate * grad_x
    R_y_new = R_y + learn_rate * grad_y

    R_x_new = np.clip(R_x_new, -1, 1)
    R_y_new = np.clip(R_y_new, -1, 1)
    
    R_x = R_x_new
    R_y = R_y_new

    fid_history.append(F)

    if i % 100 == 0:
        print(f"Iter {i}, Fidelity = {F:.6f}, lr = {learn_rate:.6e}")

    if F > 0.999:
        print("Converged.")
        break

print("Final fidelity:", F)
print("Final matrix: \n", Ufinal)
print(R_x, R_y)

Target matrix: 
 [[1 0 0 0]
 [0 1 0 0]
 [0 0 0 1]
 [0 0 1 0]]
Iter 0, Fidelity = 0.369912, lr = 1.000000e-02
Iter 100, Fidelity = 0.430625, lr = 1.000000e-02
Iter 200, Fidelity = 0.462704, lr = 1.000000e-02
Iter 300, Fidelity = 0.476742, lr = 1.000000e-02
Iter 400, Fidelity = 0.482712, lr = 1.000000e-02
Iter 500, Fidelity = 0.485411, lr = 1.000000e-02
Iter 600, Fidelity = 0.486751, lr = 1.000000e-02
Iter 700, Fidelity = 0.487477, lr = 1.000000e-02
Iter 800, Fidelity = 0.487902, lr = 1.000000e-02
Iter 900, Fidelity = 0.488164, lr = 1.000000e-02
Iter 1000, Fidelity = 0.488335, lr = 1.000000e-02
Iter 1100, Fidelity = 0.488451, lr = 1.000000e-02
Iter 1200, Fidelity = 0.488534, lr = 1.000000e-02
Iter 1300, Fidelity = 0.488596, lr = 1.000000e-02
Iter 1400, Fidelity = 0.488645, lr = 1.000000e-02
Iter 1500, Fidelity = 0.488687, lr = 1.000000e-02
Iter 1600, Fidelity = 0.488724, lr = 1.000000e-02
Iter 1700, Fidelity = 0.488758, lr = 1.000000e-02
Iter 1800, Fidelity = 0.488791, lr = 1.000000e-02


In [61]:
export_to_json_GRAPE("TestGRAPE_cnot","1stpulse_cnot",F,T,N,R_x,R_y,"andreroque")

Pulse file saved to TestGRAPE_cnot


In [48]:
# learning loop

np.random.seed(1234)
U_target = U_cnot
print("Target matrix: \n", U_target)

J = 696
T = 200e-6
N = 200  # maybe add more divisions?
dt = T/N

# initialize
R_x = 0.2 * np.random.rand(N,1)
R_y = 0.2 * np.random.rand(N,1)

learn_rate = 0.01
max_iter = 5000
fid_history = []

for i in range(max_iter):

    F, grad_x, grad_y, Ufinal = grape_grad(R_x, R_y, U_target)

    # update
    R_x_new = R_x + learn_rate * grad_x
    R_y_new = R_y + learn_rate * grad_y

    R_x_new = np.clip(R_x_new, -1, 1)
    R_y_new = np.clip(R_y_new, -1, 1)
    
    R_x = R_x_new
    R_y = R_y_new

    fid_history.append(F)

    if i % 100 == 0:
        print(f"Iter {i}, Fidelity = {F:.6f}, lr = {learn_rate:.6e}")

    if F > 0.999:
        print("Converged.")
        break

print("Final fidelity:", F)
print("Final matrix: \n", Ufinal)
    

Target matrix: 
 [[1 0 0 0]
 [0 1 0 0]
 [0 0 0 1]
 [0 0 1 0]]
Iter 0, Fidelity = 0.375753, lr = 1.000000e-02
Iter 100, Fidelity = 0.408211, lr = 1.000000e-02
Iter 200, Fidelity = 0.433263, lr = 1.000000e-02
Iter 300, Fidelity = 0.451204, lr = 1.000000e-02
Iter 400, Fidelity = 0.463415, lr = 1.000000e-02
Iter 500, Fidelity = 0.471500, lr = 1.000000e-02
Iter 600, Fidelity = 0.476808, lr = 1.000000e-02
Iter 700, Fidelity = 0.480309, lr = 1.000000e-02
Iter 800, Fidelity = 0.482653, lr = 1.000000e-02
Iter 900, Fidelity = 0.484254, lr = 1.000000e-02
Iter 1000, Fidelity = 0.485371, lr = 1.000000e-02
Iter 1100, Fidelity = 0.486170, lr = 1.000000e-02
Iter 1200, Fidelity = 0.486755, lr = 1.000000e-02
Iter 1300, Fidelity = 0.487191, lr = 1.000000e-02
Iter 1400, Fidelity = 0.487523, lr = 1.000000e-02
Iter 1500, Fidelity = 0.487780, lr = 1.000000e-02
Iter 1600, Fidelity = 0.487982, lr = 1.000000e-02
Iter 1700, Fidelity = 0.488144, lr = 1.000000e-02
Iter 1800, Fidelity = 0.488275, lr = 1.000000e-02


In [49]:
# learning loop

np.random.seed(1234)
U_target = U_cnot
print("Target matrix: \n", U_target)

J = 696
T = 400e-6
N = 200  # maybe add more divisions?
dt = T/N

# initialize
R_x = 0.2 * np.random.rand(N,1)
R_y = 0.2 * np.random.rand(N,1)

learn_rate = 0.01
max_iter = 5000
fid_history = []

for i in range(max_iter):

    F, grad_x, grad_y, Ufinal = grape_grad(R_x, R_y, U_target)

    # update
    R_x_new = R_x + learn_rate * grad_x
    R_y_new = R_y + learn_rate * grad_y

    R_x_new = np.clip(R_x_new, -1, 1)
    R_y_new = np.clip(R_y_new, -1, 1)
    
    R_x = R_x_new
    R_y = R_y_new

    fid_history.append(F)

    if i % 100 == 0:
        print(f"Iter {i}, Fidelity = {F:.6f}, lr = {learn_rate:.6e}")

    if F > 0.999:
        print("Converged.")
        break

print("Final fidelity:", F)
print("Final matrix: \n", Ufinal)
    

Target matrix: 
 [[1 0 0 0]
 [0 1 0 0]
 [0 0 0 1]
 [0 0 1 0]]
Iter 0, Fidelity = 0.130653, lr = 1.000000e-02
Iter 100, Fidelity = 0.223383, lr = 1.000000e-02
Iter 200, Fidelity = 0.365901, lr = 1.000000e-02
Iter 300, Fidelity = 0.460236, lr = 1.000000e-02
Iter 400, Fidelity = 0.496205, lr = 1.000000e-02
Iter 500, Fidelity = 0.514795, lr = 1.000000e-02
Iter 600, Fidelity = 0.526194, lr = 1.000000e-02
Iter 700, Fidelity = 0.532585, lr = 1.000000e-02
Iter 800, Fidelity = 0.535923, lr = 1.000000e-02
Iter 900, Fidelity = 0.537710, lr = 1.000000e-02
Iter 1000, Fidelity = 0.538777, lr = 1.000000e-02
Iter 1100, Fidelity = 0.539513, lr = 1.000000e-02
Iter 1200, Fidelity = 0.540087, lr = 1.000000e-02
Iter 1300, Fidelity = 0.540576, lr = 1.000000e-02
Iter 1400, Fidelity = 0.541010, lr = 1.000000e-02
Iter 1500, Fidelity = 0.541408, lr = 1.000000e-02
Iter 1600, Fidelity = 0.541776, lr = 1.000000e-02
Iter 1700, Fidelity = 0.542121, lr = 1.000000e-02
Iter 1800, Fidelity = 0.542444, lr = 1.000000e-02


perm

In [50]:
# learning loop

np.random.seed(1234)
U_target = U_perm
print("Target matrix: \n", U_target)

J = 696
T = 200e-6
N = 100  # maybe add more divisions?
dt = T/N

# initialize
R_x = 0.2 * np.random.rand(N,1)
R_y = 0.2 * np.random.rand(N,1)

learn_rate = 0.01
max_iter = 5000
fid_history = []

for i in range(max_iter):

    F, grad_x, grad_y, Ufinal = grape_grad(R_x, R_y, U_target)

    # update
    R_x_new = R_x + learn_rate * grad_x
    R_y_new = R_y + learn_rate * grad_y

    R_x_new = np.clip(R_x_new, -1, 1)
    R_y_new = np.clip(R_y_new, -1, 1)
    
    R_x = R_x_new
    R_y = R_y_new

    fid_history.append(F)

    if i % 100 == 0:
        print(f"Iter {i}, Fidelity = {F:.6f}, lr = {learn_rate:.6e}")

    if F > 0.999:
        print("Converged.")
        break

print("Final fidelity:", F)
print("Final matrix: \n", Ufinal)
    

Target matrix: 
 [[1 0 0 0]
 [0 0 0 1]
 [0 1 0 0]
 [0 0 1 0]]
Iter 0, Fidelity = 0.229084, lr = 1.000000e-02
Iter 100, Fidelity = 0.258659, lr = 1.000000e-02
Iter 200, Fidelity = 0.286372, lr = 1.000000e-02
Iter 300, Fidelity = 0.311628, lr = 1.000000e-02
Iter 400, Fidelity = 0.334404, lr = 1.000000e-02
Iter 500, Fidelity = 0.354786, lr = 1.000000e-02
Iter 600, Fidelity = 0.372785, lr = 1.000000e-02
Iter 700, Fidelity = 0.388408, lr = 1.000000e-02
Iter 800, Fidelity = 0.401733, lr = 1.000000e-02
Iter 900, Fidelity = 0.412900, lr = 1.000000e-02
Iter 1000, Fidelity = 0.422095, lr = 1.000000e-02
Iter 1100, Fidelity = 0.429527, lr = 1.000000e-02
Iter 1200, Fidelity = 0.435427, lr = 1.000000e-02
Iter 1300, Fidelity = 0.440035, lr = 1.000000e-02
Iter 1400, Fidelity = 0.443585, lr = 1.000000e-02
Iter 1500, Fidelity = 0.446291, lr = 1.000000e-02
Iter 1600, Fidelity = 0.448340, lr = 1.000000e-02
Iter 1700, Fidelity = 0.449886, lr = 1.000000e-02
Iter 1800, Fidelity = 0.451053, lr = 1.000000e-02


In [51]:
# learning loop

np.random.seed(1234)
U_target = U_perm
print("Target matrix: \n", U_target)

J = 696
T = 200e-6
N = 200  # maybe add more divisions?
dt = T/N

# initialize
R_x = 0.2 * np.random.rand(N,1)
R_y = 0.2 * np.random.rand(N,1)

learn_rate = 0.01
max_iter = 5000
fid_history = []

for i in range(max_iter):

    F, grad_x, grad_y, Ufinal = grape_grad(R_x, R_y, U_target)

    # update
    R_x_new = R_x + learn_rate * grad_x
    R_y_new = R_y + learn_rate * grad_y

    R_x_new = np.clip(R_x_new, -1, 1)
    R_y_new = np.clip(R_y_new, -1, 1)
    
    R_x = R_x_new
    R_y = R_y_new

    fid_history.append(F)

    if i % 100 == 0:
        print(f"Iter {i}, Fidelity = {F:.6f}, lr = {learn_rate:.6e}")

    if F > 0.999:
        print("Converged.")
        break

print("Final fidelity:", F)
print("Final matrix: \n", Ufinal)
    

Target matrix: 
 [[1 0 0 0]
 [0 0 0 1]
 [0 1 0 0]
 [0 0 1 0]]
Iter 0, Fidelity = 0.235927, lr = 1.000000e-02
Iter 100, Fidelity = 0.250464, lr = 1.000000e-02

Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)

Iter 200, Fidelity = 0.264618, lr = 1.000000e-02
Iter 300, Fidelity = 0.278280, lr = 1.000000e-02
Iter 400, Fidelity = 0.291391, lr = 1.000000e-02
Iter 500, Fidelity = 0.303933, lr = 1.000000e-02
Iter 600, Fidelity = 0.315909, lr = 1.000000e-02
Iter 700, Fidelity = 0.327328, lr = 1.000000e-02
Iter 800, Fidelity = 0.338195, lr = 1.000000e-02
Iter 900, Fidelity = 0.348505, lr = 1.000000e-02
Iter 1000, Fidelity = 0.358247, lr = 1.000000e-02
Iter 1100, Fidelity = 0.367410, lr = 1.000000e-02
Iter 1200, Fidelity = 0.375985, lr = 1.000000e-02
Iter 1300, Fidelity = 0.383968, lr = 1.000000e-02
Iter 1400, Fidelity = 0.391366, lr = 1.000000e-02
Iter 1500, Fidelity = 0.398186, lr = 1.000000e-02
Iter 1600, Fidelity = 0.404446, lr = 

In [52]:
# learning loop

np.random.seed(1234)
U_target = U_perm
print("Target matrix: \n", U_target)

J = 696
T = 400e-6
N = 200  # maybe add more divisions?
dt = T/N

# initialize
R_x = 0.2 * np.random.rand(N,1)
R_y = 0.2 * np.random.rand(N,1)

learn_rate = 0.01
max_iter = 5000
fid_history = []

for i in range(max_iter):

    F, grad_x, grad_y, Ufinal = grape_grad(R_x, R_y, U_target)

    # update
    R_x_new = R_x + learn_rate * grad_x
    R_y_new = R_y + learn_rate * grad_y

    R_x_new = np.clip(R_x_new, -1, 1)
    R_y_new = np.clip(R_y_new, -1, 1)
    
    R_x = R_x_new
    R_y = R_y_new

    fid_history.append(F)

    if i % 100 == 0:
        print(f"Iter {i}, Fidelity = {F:.6f}, lr = {learn_rate:.6e}")

    if F > 0.999:
        print("Converged.")
        break

print("Final fidelity:", F)
print("Final matrix: \n", Ufinal)
    

Target matrix: 
 [[1 0 0 0]
 [0 0 0 1]
 [0 1 0 0]
 [0 0 1 0]]
Iter 0, Fidelity = 0.288678, lr = 1.000000e-02
Iter 100, Fidelity = 0.294772, lr = 1.000000e-02
Iter 200, Fidelity = 0.301082, lr = 1.000000e-02
Iter 300, Fidelity = 0.311262, lr = 1.000000e-02
Iter 400, Fidelity = 0.334874, lr = 1.000000e-02
Iter 500, Fidelity = 0.386020, lr = 1.000000e-02
Iter 600, Fidelity = 0.449186, lr = 1.000000e-02
Iter 700, Fidelity = 0.490176, lr = 1.000000e-02
Iter 800, Fidelity = 0.509674, lr = 1.000000e-02
Iter 900, Fidelity = 0.518339, lr = 1.000000e-02
Iter 1000, Fidelity = 0.522265, lr = 1.000000e-02
Iter 1100, Fidelity = 0.524209, lr = 1.000000e-02
Iter 1200, Fidelity = 0.525334, lr = 1.000000e-02
Iter 1300, Fidelity = 0.526107, lr = 1.000000e-02
Iter 1400, Fidelity = 0.526718, lr = 1.000000e-02
Iter 1500, Fidelity = 0.527243, lr = 1.000000e-02
Iter 1600, Fidelity = 0.527716, lr = 1.000000e-02
Iter 1700, Fidelity = 0.528152, lr = 1.000000e-02
Iter 1800, Fidelity = 0.528558, lr = 1.000000e-02


In [53]:
# learning loop

np.random.seed(1234)
U_target = U_perm
print("Target matrix: \n", U_target)

J = 696
T = 400e-6
N = 400  # maybe add more divisions?
dt = T/N

# initialize
R_x = 0.2 * np.random.rand(N,1)
R_y = 0.2 * np.random.rand(N,1)

learn_rate = 0.01
max_iter = 5000
fid_history = []

for i in range(max_iter):

    F, grad_x, grad_y, Ufinal = grape_grad(R_x, R_y, U_target)

    # update
    R_x_new = R_x + learn_rate * grad_x
    R_y_new = R_y + learn_rate * grad_y

    R_x_new = np.clip(R_x_new, -1, 1)
    R_y_new = np.clip(R_y_new, -1, 1)
    
    R_x = R_x_new
    R_y = R_y_new

    fid_history.append(F)

    if i % 100 == 0:
        print(f"Iter {i}, Fidelity = {F:.6f}, lr = {learn_rate:.6e}")

    if F > 0.999:
        print("Converged.")
        break

print("Final fidelity:", F)
print("Final matrix: \n", Ufinal)
    

Target matrix: 
 [[1 0 0 0]
 [0 0 0 1]
 [0 1 0 0]
 [0 0 1 0]]
Iter 0, Fidelity = 0.290901, lr = 1.000000e-02
Iter 100, Fidelity = 0.295367, lr = 1.000000e-02
Iter 200, Fidelity = 0.299806, lr = 1.000000e-02
Iter 300, Fidelity = 0.305048, lr = 1.000000e-02
Iter 400, Fidelity = 0.312249, lr = 1.000000e-02
Iter 500, Fidelity = 0.323119, lr = 1.000000e-02
Iter 600, Fidelity = 0.339871, lr = 1.000000e-02
Iter 700, Fidelity = 0.364214, lr = 1.000000e-02
Iter 800, Fidelity = 0.395159, lr = 1.000000e-02
Iter 900, Fidelity = 0.427895, lr = 1.000000e-02
Iter 1000, Fidelity = 0.456673, lr = 1.000000e-02
Iter 1100, Fidelity = 0.478652, lr = 1.000000e-02
Iter 1200, Fidelity = 0.494105, lr = 1.000000e-02
Iter 1300, Fidelity = 0.504541, lr = 1.000000e-02
Iter 1400, Fidelity = 0.511465, lr = 1.000000e-02
Iter 1500, Fidelity = 0.516033, lr = 1.000000e-02
Iter 1600, Fidelity = 0.519057, lr = 1.000000e-02
Iter 1700, Fidelity = 0.521084, lr = 1.000000e-02
Iter 1800, Fidelity = 0.522474, lr = 1.000000e-02


series of hard pulses

In [39]:
# Gradient Ascent - GRAPE

# a bit different from the basic SMP, the parameter phi is omitted, collapsed into the amplitude which now differs for each orthogonal direction x, y

U_cnot = np.array([[1,0,0,0],
                   [0,1,0,0],
                   [0,0,0,1],
                   [0,0,1,0]])

U_perm = np.array([[1,0,0,0],
                   [0,0,0,1],
                   [0,1,0,0],
                   [0,0,1,0]])

np.random.seed(1234)
U_target = U
print(U_target)

# constants
n = 2
d = 2**n
w0 = 6250
w1_max = 6250
J = 696
T = 200e-6
N = 100  # maybe add more divisions?
dt = T/N

# fixed hamiltonians
H_J = 2*np.pi*J*(np.kron(Z,Z)/4)
H_x = 2*np.pi*w1_max*(Ix1+Ix2)
H_y = 2*np.pi*w1_max*(Iy1+Iy2)

# initialize
R_x = 0.2 * np.random.rand(N,1)
R_y = 0.2 * np.random.rand(N,1)

def p_calc(R_x,R_y):
    P_list = []
    P = U_target.copy()

    for k in reversed(range(N)):
        H = H_J + R_x[k]*H_x + R_y[k]*H_y
        Uk = scipy.linalg.expm(-1j * H * dt)
        P = Uk.conj().T @ P
        P_list.insert(0, P)

    return P_list

def x_calc(R_x,R_y):
    X_list = []
    U = np.eye(d, dtype=complex)

    for k in range(N):
        H = H_J + R_x[k]*H_x + R_y[k]*H_y
        Uk = scipy.linalg.expm(-1j * H * dt)
        U = Uk @ U
        X_list.append(U)

    return X_list, U

def fidelity(U):
    return np.abs(np.trace(U_target.conj().T @ U))/d

def grape_grad(R_x, R_y):
    
    X_list, U_final = x_calc(R_x, R_y)
    P_list = p_calc(R_x, R_y)
    
    phi = np.trace(U_target.conj().T @ U_final)
    F0 = fidelity(U_final)
    
    grad_x = np.zeros((N,1))
    grad_y = np.zeros((N,1))
    
    for k in range(N):
        Xk = X_list[k]
        Pk = P_list[k]
        
        dX = -1j * dt * H_x @ Xk
        dY = -1j * dt * H_y @ Xk
        
        grad_x[k] = (2/d) * np.real(
            np.trace(Pk.conj().T @ dX) * np.conj(phi)
        )
        
        grad_y[k] = (2/d) * np.real(
            np.trace(Pk.conj().T @ dY) * np.conj(phi)
        )
        
    return F0, grad_x, grad_y, U_final

    
# learning loop

learn_rate = 0.01
max_iter = 10000
fid_history = []

for i in range(max_iter):

    F, grad_x, grad_y, Ufinal = grape_grad(R_x, R_y)

    # update
    R_x_new = R_x + learn_rate * grad_x
    R_y_new = R_y + learn_rate * grad_y

    R_x_new = np.clip(R_x_new, -1, 1)
    R_y_new = np.clip(R_y_new, -1, 1)

    # --- Optional: monotonicity check (important!) ---
    F_new, _, _, _ = grape_grad(R_x_new, R_y_new)

    if F_new >= F:
        R_x = R_x_new
        R_y = R_y_new
        F = F_new
    else:
        # reduce step size if fidelity decreases
        learn_rate *= 0.5

    fid_history.append(F)

    if i % 100 == 0:
        print(f"Iter {i}, Fidelity = {F:.6f}, lr = {learn_rate:.6e}")

    if F > 0.999:
        print("Converged.")
        break

print("Final fidelity:", F)
print(Ufinal)

    

[[ 0.8392-0.1136j  0.3991-0.0133j -0.2826+0.0629j -0.1991-0.j    ]
 [-0.3088+0.2879j  0.6379-0.4843j  0.0969-0.107j  -0.3259+0.2307j]
 [ 0.1646+0.2581j  0.0969+0.107j   0.5435+0.7115j  0.1871+0.221j ]
 [-0.1047-0.j      0.4162-0.0706j -0.305 +0.0258j  0.8392-0.1136j]]
Iter 0, Fidelity = 0.411075, lr = 1.000000e-02
Iter 100, Fidelity = 0.683140, lr = 1.000000e-02
Iter 200, Fidelity = 0.740628, lr = 1.000000e-02
Iter 300, Fidelity = 0.744831, lr = 1.000000e-02
Iter 400, Fidelity = 0.745305, lr = 1.000000e-02
Iter 500, Fidelity = 0.745531, lr = 1.000000e-02
Iter 600, Fidelity = 0.745730, lr = 1.000000e-02
Iter 700, Fidelity = 0.745919, lr = 1.000000e-02
Iter 800, Fidelity = 0.746099, lr = 1.000000e-02
Iter 900, Fidelity = 0.746272, lr = 1.000000e-02
Iter 1000, Fidelity = 0.746438, lr = 1.000000e-02
Iter 1100, Fidelity = 0.746598, lr = 1.000000e-02
Iter 1200, Fidelity = 0.746750, lr = 1.000000e-02
Iter 1300, Fidelity = 0.746896, lr = 1.000000e-02
Iter 1400, Fidelity = 0.747036, lr = 1.0000

In [ ]:
# Gradient Ascent - GRAPE

# a bit different from the basic SMP, the parameter phi is omitted, collapsed into the amplitude which now differs for each orthogonal direction x, y

U_cnot = np.array([[1,0,0,0],
                   [0,1,0,0],
                   [0,0,0,1],
                   [0,0,1,0]])

U_perm = np.array([[1,0,0,0],
                   [0,0,0,1],
                   [0,1,0,0],
                   [0,0,1,0]])

np.random.seed(1234)
U_target = U
print(U_target)

# constants
n = 2
d = 2**n
w0 = 6250
w1_max = 6250
J = 696
T = 400e-6
N = 100  # maybe add more divisions?
dt = T/N

# fixed hamiltonians
H_J = 2*np.pi*J*(np.kron(Z,Z)/4)
H_x = 2*np.pi*w1_max*(Ix1+Ix2)
H_y = 2*np.pi*w1_max*(Iy1+Iy2)

# initialize
R_x = 0.2 * np.random.rand(N,1)
R_y = 0.2 * np.random.rand(N,1)

def p_calc(R_x,R_y):
    P_list = []
    P = U_target.copy()

    for k in reversed(range(N)):
        H = H_J + R_x[k]*H_x + R_y[k]*H_y
        Uk = scipy.linalg.expm(-1j * H * dt)
        P = Uk.conj().T @ P
        P_list.insert(0, P)

    return P_list

def x_calc(R_x,R_y):
    X_list = []
    U = np.eye(d, dtype=complex)

    for k in range(N):
        H = H_J + R_x[k]*H_x + R_y[k]*H_y
        Uk = scipy.linalg.expm(-1j * H * dt)
        U = Uk @ U
        X_list.append(U)

    return X_list, U

def fidelity(U):
    return np.abs(np.trace(U_target.conj().T @ U))/d

def grape_grad(R_x, R_y):
    
    X_list, U_final = x_calc(R_x, R_y)
    P_list = p_calc(R_x, R_y)
    
    phi = np.trace(U_target.conj().T @ U_final)
    F0 = fidelity(U_final)
    
    grad_x = np.zeros((N,1))
    grad_y = np.zeros((N,1))
    
    k = range(N)
    
    for k in range(N):
        Xk = X_list[k]
        Pk = P_list[k]
        
        dX = -1j * dt * H_x @ Xk
        grad_x[k] = (2/d) * np.real(
            np.trace(Pk.conj().T @ dX) * np.conj(phi)
        )
        
        dY = -1j * dt * H_y @ Xk
        grad_y[k] = (2/d) * np.real(
            np.trace(Pk.conj().T @ dY) * np.conj(phi)
        )
        
    return F0, grad_x, grad_y, U_final

    
# learning loop

learn_rate = 0.01
max_iter = 10000
fid_history = []

for i in range(max_iter):

    F, grad_x, grad_y, Ufinal = grape_grad(R_x, R_y)

    # update
    R_x_new = R_x + learn_rate * grad_x
    R_y_new = R_y + learn_rate * grad_y

    R_x_new = np.clip(R_x_new, -1, 1)
    R_y_new = np.clip(R_y_new, -1, 1)

    # --- Optional: monotonicity check (important!) ---
    F_new, _, _, _ = grape_grad(R_x_new, R_y_new)

    if F_new >= F:
        R_x = R_x_new
        R_y = R_y_new
        F = F_new
    else:
        # reduce step size if fidelity decreases
        learn_rate *= 0.5

    fid_history.append(F)

    if i % 100 == 0:
        print(f"Iter {i}, Fidelity = {F:.6f}, lr = {learn_rate:.6e}")

    if F > 0.999:
        print("Converged.")
        break

print("Final fidelity:", F)
print(Ufinal)

    

[[ 0.8392-0.1136j  0.3991-0.0133j -0.2826+0.0629j -0.1991-0.j    ]
 [-0.3088+0.2879j  0.6379-0.4843j  0.0969-0.107j  -0.3259+0.2307j]
 [ 0.1646+0.2581j  0.0969+0.107j   0.5435+0.7115j  0.1871+0.221j ]
 [-0.1047-0.j      0.4162-0.0706j -0.305 +0.0258j  0.8392-0.1136j]]
Iter 0, Fidelity = 0.058989, lr = 1.000000e-02
Iter 100, Fidelity = 0.063069, lr = 1.000000e-02
Iter 200, Fidelity = 0.068880, lr = 1.000000e-02
Iter 300, Fidelity = 0.076519, lr = 1.000000e-02
Iter 400, Fidelity = 0.083874, lr = 1.000000e-02
Iter 500, Fidelity = 0.089116, lr = 1.000000e-02
Iter 600, Fidelity = 0.092366, lr = 1.000000e-02
Iter 700, Fidelity = 0.094439, lr = 1.000000e-02
Iter 800, Fidelity = 0.095938, lr = 1.000000e-02
Iter 900, Fidelity = 0.097177, lr = 1.000000e-02
Iter 1000, Fidelity = 0.098308, lr = 1.000000e-02
Iter 1100, Fidelity = 0.099406, lr = 1.000000e-02
Iter 1200, Fidelity = 0.100514, lr = 1.000000e-02
Iter 1300, Fidelity = 0.101659, lr = 1.000000e-02
Iter 1400, Fidelity = 0.102864, lr = 1.0000

In [41]:
# Gradient Ascent - GRAPE

# a bit different from the basic SMP, the parameter phi is omitted, collapsed into the amplitude which now differs for each orthogonal direction x, y

U_cnot = np.array([[1,0,0,0],
                   [0,1,0,0],
                   [0,0,0,1],
                   [0,0,1,0]])

U_perm = np.array([[1,0,0,0],
                   [0,0,0,1],
                   [0,1,0,0],
                   [0,0,1,0]])

np.random.seed(1234)
U_target = U
print(U_target)

# constants
n = 2
d = 2**n
w0 = 6250
w1_max = 6250
J = 696
T = 400e-6
N = 300  # maybe add more divisions?
dt = T/N

# fixed hamiltonians
H_J = 2*np.pi*J*(np.kron(Z,Z)/4)
H_x = 2*np.pi*w1_max*(Ix1+Ix2)
H_y = 2*np.pi*w1_max*(Iy1+Iy2)

# initialize
R_x = 0.2 * np.random.rand(N,1)
R_y = 0.2 * np.random.rand(N,1)

def p_calc(R_x,R_y):
    P_list = []
    P = U_target.copy()

    for k in reversed(range(N)):
        H = H_J + R_x[k]*H_x + R_y[k]*H_y
        Uk = scipy.linalg.expm(-1j * H * dt)
        P = Uk.conj().T @ P
        P_list.insert(0, P)

    return P_list

def x_calc(R_x,R_y):
    X_list = []
    U = np.eye(d, dtype=complex)

    for k in range(N):
        H = H_J + R_x[k]*H_x + R_y[k]*H_y
        Uk = scipy.linalg.expm(-1j * H * dt)
        U = Uk @ U
        X_list.append(U)

    return X_list, U

def fidelity(U):
    return np.abs(np.trace(U_target.conj().T @ U))/d

def grape_grad(R_x, R_y):
    
    X_list, U_final = x_calc(R_x, R_y)
    P_list = p_calc(R_x, R_y)
    
    phi = np.trace(U_target.conj().T @ U_final)
    F0 = fidelity(U_final)
    
    grad_x = np.zeros((N,1))
    grad_y = np.zeros((N,1))
    
    for k in range(N):
        Xk = X_list[k]
        Pk = P_list[k]
        
        dX = -1j * dt * H_x @ Xk
        dY = -1j * dt * H_y @ Xk
        
        grad_x[k] = (2/d) * np.real(
            np.trace(Pk.conj().T @ dX) * np.conj(phi)
        )
        
        grad_y[k] = (2/d) * np.real(
            np.trace(Pk.conj().T @ dY) * np.conj(phi)
        )
        
    return F0, grad_x, grad_y, U_final

    
# learning loop

learn_rate = 0.01
max_iter = 10000
fid_history = []

for i in range(max_iter):

    F, grad_x, grad_y, Ufinal = grape_grad(R_x, R_y)

    # update
    R_x_new = R_x + learn_rate * grad_x
    R_y_new = R_y + learn_rate * grad_y

    R_x_new = np.clip(R_x_new, -1, 1)
    R_y_new = np.clip(R_y_new, -1, 1)

    # --- Optional: monotonicity check (important!) ---
    F_new, _, _, _ = grape_grad(R_x_new, R_y_new)

    if F_new >= F:
        R_x = R_x_new
        R_y = R_y_new
        F = F_new
    else:
        # reduce step size if fidelity decreases
        learn_rate *= 0.5

    fid_history.append(F)

    if i % 100 == 0:
        print(f"Iter {i}, Fidelity = {F:.6f}, lr = {learn_rate:.6e}")

    if F > 0.999:
        print("Converged.")
        break

print("Final fidelity:", F)
print(Ufinal)

    

[[ 0.8392-0.1136j  0.3991-0.0133j -0.2826+0.0629j -0.1991-0.j    ]
 [-0.3088+0.2879j  0.6379-0.4843j  0.0969-0.107j  -0.3259+0.2307j]
 [ 0.1646+0.2581j  0.0969+0.107j   0.5435+0.7115j  0.1871+0.221j ]
 [-0.1047-0.j      0.4162-0.0706j -0.305 +0.0258j  0.8392-0.1136j]]
Iter 0, Fidelity = 0.055148, lr = 1.000000e-02
Iter 100, Fidelity = 0.056821, lr = 1.000000e-02
Iter 200, Fidelity = 0.059224, lr = 1.000000e-02
Iter 300, Fidelity = 0.063234, lr = 1.000000e-02
Iter 400, Fidelity = 0.071047, lr = 1.000000e-02
Iter 500, Fidelity = 0.088642, lr = 1.000000e-02
Iter 600, Fidelity = 0.134573, lr = 1.000000e-02
Iter 700, Fidelity = 0.278859, lr = 1.000000e-02
Iter 800, Fidelity = 0.617132, lr = 1.000000e-02
Iter 900, Fidelity = 0.714141, lr = 1.000000e-02
Iter 1000, Fidelity = 0.718840, lr = 1.000000e-02
Iter 1100, Fidelity = 0.719953, lr = 1.000000e-02
Iter 1200, Fidelity = 0.720917, lr = 1.000000e-02
Iter 1300, Fidelity = 0.721834, lr = 1.000000e-02
Iter 1400, Fidelity = 0.722707, lr = 1.0000